In [1]:
# We always start with a dataset to train on. Let's download the tiny shakespeare dataset
#!wget https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt

In [2]:
with open("input.txt", "r", encoding="utf-8") as f:
    text = f.read()

In [3]:
print("length of dataset in characters:", len(text))

length of dataset in characters: 1115394


In [4]:
print(text[:50])

First Citizen:
Before we proceed any further, hear


In [5]:
chars = sorted(list(set(text)))
vocab_size = len(chars)

print(''.join(chars))
print(vocab_size)



 !$&',-.3:;?ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz
65


Since the goal is to build a "character" based Language model, the smallest unit of for tokens will be characters.

# Tokenization and Training-Val split

In [6]:
# Create mapping from characters to integers
stoi = {ch:i for i, ch, in enumerate(chars)} # string to integers. Produces {'\n': 0, ' ': 1, '!': 2,...
itos = {i:ch for i, ch in enumerate(chars)} # in reverse: {0: '\n',  1: ' ',  2: '!',...

encode = lambda s: [stoi[c] for c in s] # encoder: takes a string and return list of integers
decode = lambda i: "".join([itos[n] for n in i]) # decoder: takes a list of integers and returns string

print(encode("putos "))
print(decode(encode("putos ")))

[54, 59, 58, 53, 57, 1]
putos 


Now we can encode all the data

In [7]:
import torch 
data = torch.tensor(encode(text), dtype=torch.long)
print(data.shape, data.dtype)
print(data[:100])

torch.Size([1115394]) torch.int64
tensor([18, 47, 56, 57, 58,  1, 15, 47, 58, 47, 64, 43, 52, 10,  0, 14, 43, 44,
        53, 56, 43,  1, 61, 43,  1, 54, 56, 53, 41, 43, 43, 42,  1, 39, 52, 63,
         1, 44, 59, 56, 58, 46, 43, 56,  6,  1, 46, 43, 39, 56,  1, 51, 43,  1,
        57, 54, 43, 39, 49,  8,  0,  0, 13, 50, 50, 10,  0, 31, 54, 43, 39, 49,
         6,  1, 57, 54, 43, 39, 49,  8,  0,  0, 18, 47, 56, 57, 58,  1, 15, 47,
        58, 47, 64, 43, 52, 10,  0, 37, 53, 59])


Splitting data into training and validation

In [8]:

n = int(len(data)*0.9) # First 90% will be training
train_data = data[:n]
val_data = data[n:]

# Data Loader and batches of chunks

### Things to consider.
We do not need a prediction success of 100% since we do not want to predict exactly the Shakespear text, but to generate Shakespear-like text.

Also when training a Transformer we do not use all text at once, because it is computationall expensive (given the nature of the transformer). So what is usually done is to extract random chunks of text from the training set (specific lenght, or maximum lenght; this is the block size, or context length).

In [9]:
block_size = 8
train_data[:block_size+1] # Notice the plus one

tensor([18, 47, 56, 57, 58,  1, 15, 47, 58])

The magic of training is that given a single block of training we have multiple training units.

For example. If we only take token 18 in the context window, the goal (target) is to predict 47. Example 1.

If we take 18 and 47 in the context window, the goal is to predict 56; this is the second example.

And so on and so forth we can take 18 to 47 (8 numbers in total; the size of the context) to predict 58. the eight example from a single training block.

In [11]:
x = train_data[:block_size]
y = train_data[1:block_size+1]

for t in range(block_size):
    context = x[:t+1]
    targets = y[t]
    print(f"When the context is {context}, the target: {targets}")

When the context is tensor([18]), the target: 47
When the context is tensor([18, 47]), the target: 56
When the context is tensor([18, 47, 56]), the target: 57
When the context is tensor([18, 47, 56, 57]), the target: 58
When the context is tensor([18, 47, 56, 57, 58]), the target: 1
When the context is tensor([18, 47, 56, 57, 58,  1]), the target: 15
When the context is tensor([18, 47, 56, 57, 58,  1, 15]), the target: 47
When the context is tensor([18, 47, 56, 57, 58,  1, 15, 47]), the target: 58


Having multiple examples from a single training block is not just for computational efficiency, but also because in inference we want the transformer to be able to predict having as little as "1 token" in the context, up to "8 tokens". 

So this answer one queestion I had in mind earlier. I was thinking that this way of training was just predicting the next token from the "single" previous one. That sounded not logical, but now I understood that is not the case.

There is an additional dimension (other than block size and ...) that we care about: __Batch dimension.__

For every training step (feeding/inputing data to the tensor, every forward/backward pass) we not use a single training block, but multiple; and this number is given by the batch size. Multiple training blocks or "a batch" is encapsulated into "a" tensor. This is parallel processing (one for each GPU).

In [12]:
torch.manual_seed(1337)

batch_size = 4 # how many "independent" sequences will be process in parallel?
block_size = 8 # what is the maximum context length for predictions?

def get_batch(split):
    data = train_data if split == "train" else val_data
    ix = torch.randint(len(data) - block_size, (batch_size,)) # Choosing 4 (batch_size) random position between all tokens (of the training or val dataset)
    # e.g. tensor([1078327,  453969,   41646,  671252])
    
    # Defining the "batch" (set of blocks) as a tensor.
    x = torch.stack([data[i:i+block_size] for i in ix])
    y = torch.stack([data[i+1:i+block_size+1] for i in ix])
    return x, y

# Example of batch
xb, yb = get_batch("train")
print("input:")
print(xb.shape)
print(xb)
print("target:")
print(yb.shape)
print(yb)

print("---")

for b in range(batch_size): # batch dimension
    for t in range(block_size): # time dimension. _____¿Why is it called t dimension?_____
        context = xb[b, :t+1]
        target = yb[b, t]
        print(f"When input is {context} the target: {target}")


input:
torch.Size([4, 8])
tensor([[24, 43, 58,  5, 57,  1, 46, 43],
        [44, 53, 56,  1, 58, 46, 39, 58],
        [52, 58,  1, 58, 46, 39, 58,  1],
        [25, 17, 27, 10,  0, 21,  1, 54]])
target:
torch.Size([4, 8])
tensor([[43, 58,  5, 57,  1, 46, 43, 39],
        [53, 56,  1, 58, 46, 39, 58,  1],
        [58,  1, 58, 46, 39, 58,  1, 46],
        [17, 27, 10,  0, 21,  1, 54, 39]])
---
When input is tensor([24]) the target: 43
When input is tensor([24, 43]) the target: 58
When input is tensor([24, 43, 58]) the target: 5
When input is tensor([24, 43, 58,  5]) the target: 57
When input is tensor([24, 43, 58,  5, 57]) the target: 1
When input is tensor([24, 43, 58,  5, 57,  1]) the target: 46
When input is tensor([24, 43, 58,  5, 57,  1, 46]) the target: 43
When input is tensor([24, 43, 58,  5, 57,  1, 46, 43]) the target: 39
When input is tensor([44]) the target: 53
When input is tensor([44, 53]) the target: 56
When input is tensor([44, 53, 56]) the target: 1
When input is tensor([

Using n-gram (bigram in this case) prediction model, because is the simplest token prediction model (predict the next from the last element)

In [13]:
xb

tensor([[24, 43, 58,  5, 57,  1, 46, 43],
        [44, 53, 56,  1, 58, 46, 39, 58],
        [52, 58,  1, 58, 46, 39, 58,  1],
        [25, 17, 27, 10,  0, 21,  1, 54]])

In [14]:
import torch
import torch.nn as nn
from torch.nn import functional as F
torch.manual_seed(1337)

class BigramLanguageModel(nn.Module):
    
    def __init__(self, vocab_size):
        super().__init__() ## Code to call (class) constructor of the parent class (nn.Module)
        # each token directly reads off the logits for the next token from a lookup table
        self.token_embedding_table = nn.Embedding(vocab_size, vocab_size) # Creates a lookup table (matrix) of size vocab_size × vocab_size
        # I will call it Logit Table
         ## The logits are the "score" (that will be normalized to probabilities) for each 
         ## token in the token dictionary in order to "predict" the next token. 
        # THIS TABLE is the thing that the training will improve. For each word (row) we have the probability/logits of each other 
        # token (columns). The context will always be only "1" token to predict the next one.

        # At the beginning we just initialize this table.

    def forward(self, idx, targets=None):

        # idx and targets are both (B, T) tensor of integers.
         ## B: batch dim. T: time dim.
         ## idx: is a tensor of all the IDs of the input tokens
        logits = self.token_embedding_table(idx) # (B,T,C)
        ## So this code is ask the Logits Table for the logits of the inputs (the tokens being process), 
        # there are multiple inputs, the total given by the number of batches and "times" (T: length of sequence)

        
        
        # Now that we make predictions (using the logits) we can measure loss function 
        # (or quality of prediction) and a good way to do that is using the
        # "negative log likelihood loss". In pytorch is given by:
        
        if targets == None:
            loss = None
        else:
            ## Reshaping logits to fit into pytorch standard (B,C,T)
            B, T, C = logits.shape
            logits = logits.view(B*T, C) # This takes all inputs (e.g. look at xb) and 
                                        # stretch them, convert them to a single dimension: a vector (line)
                                        # But preserving the channel (C) dim as the 2nd dim.
            targets = targets.view(B*T) # or for Pytorch to guess: targets.view(-1)
            loss = F.cross_entropy(logits, targets)

        return logits, loss

    # Now that we can measure the quality of the model on some data to "generate" from the model.
    # The details of this are in previous videos of Andrej
    def generate(self, idx, max_new_tokens):
        # idx is (B, T) array of indices in the current context
        for _ in range(max_new_tokens):
            # get the predictions
            logits, loss = self(idx) # this is like "m(xb, yb)" invokes the model instance. # We don't use the loss here
            # focus only on the last time step (i.e. last t of T)
            logits = logits[:, -1, :] # becomes (B, C)
            # apply softmax to get probabilities
            probs = F.softmax(logits, dim=-1) # (B, C)
            # sample from the distribution
            idx_next = torch.multinomial(probs, num_samples=1) # (B, 1) ## This selects "randomly" a token from the vocab given 
                                                                        ## the weights from the probabilities (i.e. probs, in this example)
                                                                        # This is the prediction of the next token for each batch.
            # append sampled index to the running sequence
            idx = torch.cat((idx, idx_next), dim=1) # (B, T+1)
        return idx

    
m = BigramLanguageModel(vocab_size)
logits, loss = m(xb, yb) # This outputs the (loss and) logits  for every input (total inputs given by B times T)
print(logits.shape)
print(loss)
idx = torch.zeros((1,1), dtype=torch.long) # Time and Batch will be 1, i.e. (1,1) tensor with value 0. Remember 0 stands for "\n" in our vocab embedding
print(decode(m.generate(idx, max_new_tokens=100)[0].tolist()))



torch.Size([32, 65])
tensor(4.8786, grad_fn=<NllLossBackward0>)

SKIcLT;AcELMoTbvZv C?nq-QE33:CJqkOKH-q;:la!oiywkHjgChzbQ?u!3bLIgwevmyFJGUGp
wnYWmnxKWWev-tDqXErVKLgJ


This Bigram model implies that the tokens within the context doe not talk to each other. Because each input token makes prediction solely on itself.

## Training the model

In [15]:
# Creating PyTorch optimizer object
optimizer = torch.optim.AdamW(m.parameters(), lr=1e-3)

In [16]:
batch_size = 32
for steps in range(10_000):
    # sample batch of data
    xb, yb, = get_batch("train")

    # evaluate loss
    logits, loss = m(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()

print(loss.item())

2.382369041442871


In [17]:
print(decode(m.generate(idx=torch.zeros((1,1), dtype=torch.long), max_new_tokens=500)[0].tolist()))


lso br. ave aviasurf my, yxMPZI ivee iuedrd whar ksth y h bora s be hese, woweee; the! KI 'de, ulseecherd d o blllando;LUCEO, oraingofof win!
RIfans picspeserer hee tha,
TOFonk? me ain ckntoty ded. bo'llll st ta d:
ELIS me hurf lal y, ma dus pe athouo
BEY:! Indy; by s afreanoo adicererupa anse tecorro llaus a!
OLeneerithesinthengove fal amas trr
TI ar I t, mes, n IUSt my w, fredeeyove
THek' merer, dd
We ntem lud engitheso; cer ize helorowaginte the?
Thak orblyoruldvicee chot, p,
Bealivolde Th li


In [18]:
import torch

# Check if Apple's Metal backend is available
if torch.backends.mps.is_available():
    device = torch.device("mps")
    print("Using M1 GPU!")
else:
    device = torch.device("cpu")

Using M1 GPU!


# The mathematical trick in self-attention

In [46]:
# consider the following toy example:

torch.manual_seed(1337)
B, T, C = 4, 8, 2 # batch, time, channels
x = torch.rand(B, T, C)
x.shape

torch.Size([4, 8, 2])

In [47]:
# We want x[b,t] = mean_[i<=t] x[b,i]
xbow = torch.zeros((B,T,C)) # bow: bag of words: it is the summary of self-attention for every, batch for every token.
# RELEVANT: We want to say (for every batch) each token contains what information, i.e. the summary of its preceedings, in this case summary==mean.
for b in range(B):
    for t in range(T):
        xprev = x[b, :t+1] # (t, C)
        xbow[b,t] = torch.mean(xprev, 0) # (C), one dimension tensor; because we average over 1st dim (0)


In [21]:
x[0]

tensor([[0.0783, 0.4956],
        [0.6231, 0.4224],
        [0.2004, 0.0287],
        [0.5851, 0.6967],
        [0.1761, 0.2595],
        [0.7086, 0.5809],
        [0.0574, 0.7669],
        [0.8778, 0.2434]])

In [22]:
 xbow[0]

tensor([[0.0783, 0.4956],
        [0.3507, 0.4590],
        [0.3006, 0.3156],
        [0.3717, 0.4108],
        [0.3326, 0.3806],
        [0.3953, 0.4140],
        [0.3470, 0.4644],
        [0.4134, 0.4368]])

This does the job, but is inefficient 

In [23]:
torch.manual_seed(42)
a = torch.ones(3,3)
b = torch.randint(0, 10, (3,2)).float()
c = a @ b

print('a=')
print(a)
print('--')
print('b:')
print(b)
print('--')
print('c:')
print(c)

a=
tensor([[1., 1., 1.],
        [1., 1., 1.],
        [1., 1., 1.]])
--
b:
tensor([[2., 7.],
        [6., 4.],
        [6., 5.]])
--
c:
tensor([[14., 16.],
        [14., 16.],
        [14., 16.]])


We see that the vector that contains the information is 'b'. For a single batch (out of B) we have a tensor (matrix) of shape (T,C)
The rows are T, and columns C. T is the tokens and C is the values that a token can have.

We want to construct the "self-attention" that is that each token pays attention to its sorouding tokens, but hidding (masking) the precessing tokens (the future ones)
and the simplest way to get information from sorouding tokens is simply averging them out (in this simple case we do this, despite that we loose information. e.g. sequence of tokens).

The mathematical trick to do only make current tokens to talk to past ones is using the matrix a. Look at how a is changing

In [24]:
torch.manual_seed(42)
a = torch.tril(torch.ones(3,3)) # now we zero all (proceeding) tokens after the current one
b = torch.randint(0, 10, (3,2)).float()
c = a @ b

print('a=')
print(a)
print('--')
print('b:')
print(b)
print('--')
print('c:') # Watching a column (c) we see that is an accumulative sum of T's.
print(c)

a=
tensor([[1., 0., 0.],
        [1., 1., 0.],
        [1., 1., 1.]])
--
b:
tensor([[2., 7.],
        [6., 4.],
        [6., 5.]])
--
c:
tensor([[ 2.,  7.],
        [ 8., 11.],
        [14., 16.]])


 For matrix 'a' every columns is a token. i.e. column 1 is t_1, columns is t_2...

In [25]:
torch.manual_seed(42)
a = torch.tril(torch.ones(3,3))
b = torch.randint(0, 10, (3,2)).float()
c = a @ b

print('a=')
print(a)
print('--')
print('b:')
print(b)
print('--')
print('c:')
print(c)

a=
tensor([[1., 0., 0.],
        [1., 1., 0.],
        [1., 1., 1.]])
--
b:
tensor([[2., 7.],
        [6., 4.],
        [6., 5.]])
--
c:
tensor([[ 2.,  7.],
        [ 8., 11.],
        [14., 16.]])


__Look!__ that in our bag of words we are averaging even the current element t, not just the past ones

In [26]:
torch.manual_seed(42)
a = torch.tril(torch.ones(3,3))
#a = a / torch.tril(torch.ones(3,3)).sum(dim=1, keepdim=True)
a = a / torch.sum(a, dim=1, keepdim=True) # the commented line above is the same
b = torch.randint(0, 10, (3,2)).float()
c = a @ b

print('a=')
print(a)
print('--')
print('b:')
print(b)
print('--')
print('c:')
print(c)

a=
tensor([[1.0000, 0.0000, 0.0000],
        [0.5000, 0.5000, 0.0000],
        [0.3333, 0.3333, 0.3333]])
--
b:
tensor([[2., 7.],
        [6., 4.],
        [6., 5.]])
--
c:
tensor([[2.0000, 7.0000],
        [4.0000, 5.5000],
        [4.6667, 5.3333]])


We can see that location (3,1) is the average of the first column of b (t1, t2, t3).
Summed and divided by 3

__With the above inspiration we can do matrix multiplication to speed up the construction of bow__

In [ ]:
wei = torch.tril(torch.ones(T,T)) # wei: weights
wei = wei / wei.sum(1,keepdim=True)
wei # We can see the rows as "tokens" sub i being affected by the tokens in the columns.
# Example the second token in the input will only be affected by the first and second, and zero by the following

tensor([[1.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.5000, 0.5000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.3333, 0.3333, 0.3333, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.2500, 0.2500, 0.2500, 0.2500, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.2000, 0.2000, 0.2000, 0.2000, 0.2000, 0.0000, 0.0000, 0.0000],
        [0.1667, 0.1667, 0.1667, 0.1667, 0.1667, 0.1667, 0.0000, 0.0000],
        [0.1429, 0.1429, 0.1429, 0.1429, 0.1429, 0.1429, 0.1429, 0.0000],
        [0.1250, 0.1250, 0.1250, 0.1250, 0.1250, 0.1250, 0.1250, 0.1250]])

wei is like the matrix 'a' in the above example

In [ ]:
wei = torch.tril(torch.ones(T,T)) # wei: weights
wei = wei / wei.sum(1,keepdim=True)
xbow2 = wei @ x # (T,T) @ (B, T, C). Despite one being in R2 and the other on R3, Pytorch fixes this by cloning the first matrix and making B copies for the B dimension
# hence (T,T) @ (B, T, C) ---> (B, T, C). Does the operation of (T,T) to every batch of (B,T,C)
# e.g for B=0: this prduct is done: (T, T) @ (T,C)_ofB0
# And the result is like an updated token_embedding_table, that holds the look up table of the bigram model (probability of each channel for every token in the input) 

# NEED TO CONFIRM THE ABOVE STATEMENT.

# Let's first think about what does each value (coordinate) of x means.
# x is the input, this means is a set (batch) of phrases being processed.
# Each phrase has words/tokens. Each token is an embedding (encoded). IS THIS TRUE? 
# BECAUSE WHY THE 'C' DIMENSION IS PRESENT? MAYBE THEY ARE PROBABILITIES of the LOOKUP TABLE


In [35]:
# xbow and xbow2 are identical (elementwise):
torch.allclose(xbow, xbow2)

True

In [37]:
print(xbow[0])
print(xbow2[0])

tensor([[0.0783, 0.4956],
        [0.3507, 0.4590],
        [0.3006, 0.3156],
        [0.3717, 0.4108],
        [0.3326, 0.3806],
        [0.3953, 0.4140],
        [0.3470, 0.4644],
        [0.4134, 0.4368]])
tensor([[0.0783, 0.4956],
        [0.3507, 0.4590],
        [0.3006, 0.3156],
        [0.3717, 0.4108],
        [0.3326, 0.3806],
        [0.3953, 0.4140],
        [0.3470, 0.4644],
        [0.4134, 0.4368]])


__Version 3: Using Softmax. AHA! Maybe this is just like in the DeepBlue3 video, and we are just computing the weight of the attention (the attention itself)__

In [40]:
tril = torch.tril(torch.ones(T,T))
tril

tensor([[1., 0., 0., 0., 0., 0., 0., 0.],
        [1., 1., 0., 0., 0., 0., 0., 0.],
        [1., 1., 1., 0., 0., 0., 0., 0.],
        [1., 1., 1., 1., 0., 0., 0., 0.],
        [1., 1., 1., 1., 1., 0., 0., 0.],
        [1., 1., 1., 1., 1., 1., 0., 0.],
        [1., 1., 1., 1., 1., 1., 1., 0.],
        [1., 1., 1., 1., 1., 1., 1., 1.]])

In [42]:
wei = torch.zeros((T,T))
wei

tensor([[0., 0., 0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0., 0., 0.]])

In [ ]:
tril = torch.tril(torch.ones(T,T))
wei = torch.zeros((T,T)) 
wei = wei.masked_fill(tril==0, float('-inf')) # wei turns into tril but, the zeroes now are -inf
wei = F.softmax(wei, dim=-1) #Softmax is done along every row. Also This is equal to the lower triangular matrix that every row sums to one
xbow3 = wei @ x ## same as the others bow (bag of words)
torch.allclose(xbow, xbow3)

True

The above method is interesting because wei denotes the affinity (or interaction strength; how much of the tokens from the past we want to aggregate and average out) between the tokens. And first we begins as a matrix of only zeros, i.e. zero affinity at the beggining.

Then the mask ing line of code tells that future tokens cannot communicate, setting them to negative inf we are not aggregating noting of those tokens.

Then in another step the tokens will ask each other how if the other sorounding tokens are importat to each of them (this will depend on its value (embedding))




In [50]:
torch.allclose(xbow, xbow3)


True

In [60]:
# version 3: self-attention
torch.manual_seed(1337)
B,T,C = 4,8,32 # batch, time, channels
x = torch.rand(B,T,C)

## We do not want to transmit information of past tokens by simply averaging them out (uniformly weighted average)
## We want tokens to find some other tokens more interesting than others, i.e. give more weight
## e.g if a token is a vowel, maybe it will give more weight to consonants.
## TLTR: We want to ncorporate information from the past, but the magnitude is by a data dependent way.

## How do self-attention solves this? 
# Every single token at each position will emit two vector: 
# - Query vector. What i am looking for.
# - Key. What do I contain.

# Now the way we define "affinity" between these tokens sequence is by doing a
#  dot product between the queries and keys. E.g. my query times (dot products) the keys of all other tokens
# and that dot product now becomes "wei" !!



## Let's see a single Head (of attention) perform self-attention
head_size = 16 # hyperparameter of a head. We can see that is smaller than the embedding size, just like in the video of deepBlue3
key = nn.Linear(C, head_size, bias=False)
query = nn.Linear(C, head_size, bias=False)
value = nn.Linear(C, head_size, bias=False)
k = key(x) # (B,T,16)
q = query(x) # (B,T,16)

# Until now all the when forwarding x, all the tokens in all the positions (of x) in B,T arrangement each of them
# independently produce a k, q. Not communication has come between them (tokens of x)

# Here comes the communication (via the dot product)
wei = q @ k.transpose(-2,-1) # Careful because we cannot dot product q and k directly, we have to transpose, but also be careful with B dim.
## So thats why we only transpose the last two dimensions of k.
## So the matrix mult happening is: (B,T,16) @ (B,16,T) --> (B,T,T)
## So for every "row" of B we are going to have a (T,T) matrix giving us the affinities

tril = torch.tril(torch.ones(T, T))
# wei = torch.zeros(T,T) We comment this because the affinities will come from the dot product
wei = wei.masked_fill(tril==0, float('-inf'))
wei = F.softmax(wei, dim=-1) # 'dim=-1' tells pytorch to apply the softmax to the 'last' dim of the tensor

v = value(x)
out = wei @ v ## we are applying the weights (and summing, because thats the dot product) not to x itself (input), but to the value that each token generates
# (what each token adds). remember the example of deepblue3 of the "fluffy blue creature"
# each token 'fluffy', 'blue' adds a value to the input token beign modified to make it less generic. 

out.shape  

torch.Size([4, 8, 16])

In [ ]:
wei[:2,:,:] 
# Now we see (e.g. for 2 batches) that the weights are not equally distributed across "past tokens"

# For batch 1 the last token "0.1524" knows the context it has and the position it is in, and based on those two pieces of information 
# it creates a query ("I am looking for this stuff, e.g. I am a vowel, at 8th position, and i am looking for
#  consonants upt to 4th position.").
# Parallelly all other tokens emit a key vector which one could say "i am a consonant at position 3")
# they can both have a high magnitude in a a channel (dimension/direction) at that's why their dot product will
# give a high positive value

# Let's interpretate what x and the q,k,v vectors are.
# x is the private information for each token , e.g. i am the fifth token and i have some identity), that information is kept in vector x
# And for the purposes of this single head here is the information it transmits for each token:
# Here is what i am interested in (Q)
# here is what i have (K)
# And if you find me interesting (as a token) here is what i will communicate to you!!! This is V
 

tensor([[[1.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
         [0.4409, 0.5591, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
         [0.2975, 0.3373, 0.3652, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
         [0.2211, 0.2898, 0.2236, 0.2654, 0.0000, 0.0000, 0.0000, 0.0000],
         [0.1832, 0.2163, 0.1954, 0.2437, 0.1614, 0.0000, 0.0000, 0.0000],
         [0.1330, 0.2227, 0.1784, 0.2159, 0.1044, 0.1456, 0.0000, 0.0000],
         [0.1283, 0.1367, 0.1385, 0.1522, 0.1083, 0.1341, 0.2021, 0.0000],
         [0.1064, 0.1332, 0.1265, 0.1445, 0.0940, 0.1200, 0.1231, 0.1524]],

        [[1.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
         [0.4150, 0.5850, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
         [0.2313, 0.3588, 0.4098, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
         [0.2210, 0.2636, 0.2829, 0.2324, 0.0000, 0.0000, 0.0000, 0.0000],
         [0.1771, 0.2003, 0.2343, 0.2048, 0.1834, 0.0000, 0.0000, 0.0000],
         [0.1298, 0.181

## Notas

__Node 1__

Attention is like a communication mechanism between nodes in a directed graph. 
Every node has a vector of information and we want to aggregate via a weighted sum that points to it.

In the self-attention context not all nodes point to all others. Remember the causal masking and that there are only 8 nodes at a time.
- The first node is pointed only by itself, the second node
- The second is pointed by the first one and itself
- up to the 8th pointed by itself and every other node

We have to notice that attention can be applied by any directed graph, not just this autogregressive one; because is just a communication mechanism between nodes

__Note 2__

There is not notion of space. The vectors (tokens embeddings) do not know where they are. If they are the first, third or last words, you have to explicitly tell them by creating a sense of "space" (positional encoding).

This is not like CNN where space is inherent by the architecture.

__Note 3__

The elements in the batch dimension never talks to each other, each batch is proceeseed independently (via matrix multiplication):

```python
wei = q @ k. transpose(-2, -1) # (B, T, 16) @ (B, 16, T) →→→> (B, T, T)
```


The causal masking applies for this case, but for example for a "sentiment analysis" predictor we do not care that tokens talk with future tokens, __also for my research in VLA models.__ In those cases we need a encoder block of self-attention,  This means we would need to delete this line of code:

```python
wei = wei.masked_fill(tril== 0, float'-inf')) # This is a attention decoder block, not letting future talk to present
```

to let all nodes talk to each other (remove causal mask)

__Note 4__

There are many types of attention, why is this called "self" attention?
This is because the K, Q, and V come from the same source: x. i.e. the nodes are self-attending (to themselves)

But, attention is more general than that. In encoder-decoder attention we can have a case in which the queries (Q) are produced from x; but the K, V come from a separate external source, e.g. encoders blocks; that encode some context that we like to condition from.

So Cross-attention is when there is a separate source nodes where we need to pool information from to our nodes, and self-attention we just need our pool of nodes to talk to each other. The one we are modeling is self.


__Note 5__

why the formula of self-attention has a component that is the product by one over the square root of head dimension?

Because if we assume that the inputs (k,q) are randomly distributes as a (standarized normal-gaussian) then without doing that product the variance (of 'wei') is (almost) equal to the dimension of the head. That is why divide it by the square root of the head dim; to standarize it (standard deviation of 1).

That is important because will feed to softmax (function) and we prefer that wei is (fairly diffused) evenly distributed across 0, with not big numbers. 

The problem of not doing this is that softmax will start behaving as a max function, (biasing to the larger node, value).

Long story short: to make the softmax function to work better, and this implies that information across more nodes is taking into account, not only from the larger (magnitude of resulting dot products) nodes.